# Load Library

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pickle
from annoy import AnnoyIndex

C:\Users\ptrir\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Build Artifact

In [3]:
# data = pd.read_csv('data/clean/movies_drama_v2.csv')
data = pd.read_csv('../data/processed_moviesdrama.csv')
print(data.shape)
data.head()

(4905, 24)


,id,title,native-title,genre,director,format,type,country,release_date,duration,...,duration (min),release_year,rate_num,score,user_count,rating_norm,actor,combined_emb,combined_tf,poster
0,735043,When Life Gives You Tangerines,폭싹 속았수다,"Romance, Life, Drama",Kim Won Suk,Standard Series,Drama,South Korea,"Mar 7, 2025",1 hr. 2 min.,...,62,2025,13,9.3,68706,0.93,"IU, Park Bo Gum, Moon So Ri, Park Hae Joon, Ki...","When Life Gives You Tangerines Romance, Life, ...",when life gives you tangerines romance life dr...,https://i.mydramalist.com/5v8b2y_4c.jpg?v=1
1,739603,Twinkling Watermelon,반짝이는 워터멜론,"Romance, Youth, Drama, Fantasy",Son Jung Hyun,Standard Series,Drama,South Korea,"Sep 25, 2023",1 hr. 10 min.,...,70,2023,15,9.2,103873,0.92,"Ryeo Un, Choi Hyun Wook, Seol In Ah, Shin Eun ...","Twinkling Watermelon Romance, Youth, Drama, Fa...",twinkling watermelon romance youth drama fanta...,https://i.mydramalist.com/2w44jE_4c.jpg?v=1
2,49231,Move to Heaven,무브 투 헤븐: 나는 유품정리사입니다,"Life, Drama",Kim Sung Ho,Standard Series,Drama,South Korea,"May 14, 2021",52 min.,...,52,2021,18,9.1,75160,0.91,"Lee Je Hoon, Tang Jun Sang, Hong Seung Hee, Ju...","Move to Heaven Life, Drama Life, Drama Uncle-N...",move to heaven life drama life drama uncle nep...,https://i.mydramalist.com/Rle36_4c.jpg?v=1
3,702267,Weak Hero Class 1,약한영웅 Class 1,"Action, Youth, Drama",Park Dhan Hee,Standard Series,Drama,South Korea,"Nov 18, 2022",40 min.,...,40,2022,18,9.1,114265,0.91,"Park Ji Hoon, Choi Hyun Wook, Hong Kyung, Kim ...","Weak Hero Class 1 Action, Youth, Drama Action,...",weak hero class 1 action youth drama action yo...,https://i.mydramalist.com/pq2lr_4c.jpg?v=1
4,52939,Alchemy of Souls,환혼,"Action, Historical, Romance, Fantasy",Park Joon Hwa,Standard Series,Drama,South Korea,"Jun 18, 2022",1 hr. 20 min.,...,80,2022,15,9.1,108485,0.91,"Lee Jae Wook, Jung So Min, Hwang Min Hyun, Shi...","Alchemy of Souls Action, Historical, Romance, ...",alchemy of souls action historical romance fan...,https://i.mydramalist.com/Beg4z_4c.jpg?v=1


## TF-IDF

In [4]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

### Data Version 2

In [5]:
## Transform into TF-IDF Matrix
tf_matrix_v2 = vectorizer.fit_transform(data['combined_tf'])

## Calculate similarity matrix
tf_sim_v2 = cosine_similarity(tf_matrix_v2).astype(np.float32)
print(f'TF-IDF Sim dim: {tf_sim_v2.shape}')

## Save artifact
np.save('tfidf_sim_v2.npy', tf_sim_v2)

TF-IDF Sim dim: (4905, 4905)


## Embedding (MiniLM)

In [6]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

embedding = model.encode(data['combined_emb'].tolist(), normalize_embeddings=True)
print(f'Embedding dim: {embedding.shape}')

Embedding dim: (4905, 384)


In [7]:
np.save('embedding.npy', embedding)

## Annoy

In [8]:
## Define parameter value
embedding_dim = embedding.shape[1]
n_trees = 250

## Initialize index
index = AnnoyIndex(embedding_dim, 'angular')

## Add vector into index
for i, vector in enumerate(embedding):
    index.add_item(i,vector)

## Build 
index.build(n_trees)
index.save('index_movdrama.ann')
print(f"Index built: {len(embedding)} dramas, dim={embedding_dim}")

Index built: 4905 dramas, dim=384


# Testing

In [3]:
data = pd.read_csv('data/clean/movies_drama_v2.csv')
print(data.shape)

(4958, 23)


In [4]:
# Load embedding artifact
art_embedding = np.load('artifact/embedding.npy')
dim = art_embedding.shape[1]
index = AnnoyIndex(dim, 'angular')
a_annoy = index.load('artifact/index_movdrama.ann')

In [5]:
item = 47
neighbor, distance = index.get_nns_by_item(item,10,include_distances=True)

print(f"Query Drama: {data.iloc[item]['title']}")
print("\nTop 10 Recommendations:")
for idx, dist in zip(neighbor[1:], distance[1:]):
    similarity = 1 - (dist**2)/2
    print(f"{data.iloc[idx]['title']} — similarity: {similarity: .3f}")


Query Drama: Daily Dose of Sunshine

Top 10 Recommendations:
Sunshine Love — similarity:  0.624
Doctors — similarity:  0.623
D-Day — similarity:  0.609
Sunshine — similarity:  0.603
Golden Time — similarity:  0.599
Healing Camp: One World — similarity:  0.593
A Place in the Sun — similarity:  0.586
One Fine Week — similarity:  0.582
More Charming By The Day — similarity:  0.581


### TF-IDF V2

In [6]:
## Load artifact
art_tfidf_v2 = np.load('artifact/tfidf_sim_v2.npy')

## Testing
target_sim = art_tfidf_v2[item]
# Sorting
top_rec = target_sim.argsort()[::-1][1:11] 
print(f"Query Drama: {data.iloc[item]['title']}")
print("\nTop 10 Recommendations:")
for i in top_rec:
    print(f"{data.iloc[i]['title']} — similarity: {target_sim[i]: .3f}")

Query Drama: Daily Dose of Sunshine

Top 10 Recommendations:
A Poem a Day — similarity:  0.329
Heart Surgeons — similarity:  0.305
Mysterious Nurse — similarity:  0.305
The Trauma Code: Heroes on Call — similarity:  0.300
The 3rd Ward — similarity:  0.299
Blossom — similarity:  0.285
Tell Me Sick — similarity:  0.273
Dr. Romantic Season 3 — similarity:  0.270
Good Doctor — similarity:  0.268
Hospital Playlist — similarity:  0.266


## Assemble the system

In [7]:
def recommender(item, top_n):
    
    tfidf_scores = art_tfidf_v2[item]
    neighbor, distance = index.get_nns_by_item(item, 100, include_distances=True)

    # Initialize array for annoy reorder
    mini_lm = np.zeros(len(data))
    for idx,dist in zip(neighbor[1:], distance[1:]):
        mini_lm[idx] = 1 - (dist**2)/2
    
    final_scores = (
        0.3 * tfidf_scores +
        0.5 * mini_lm +
        0.2 * data['rating_norm'].values
        
    )
    final_scores[item] = -1

    top_rec = final_scores.argsort()[::-1][:top_n]
    return data.iloc[top_rec][['title', 'genre', 'score']]
    

In [19]:
a = recommender(57, 10)
a

,title,genre,score
2933,Moon Embracing the Sun,"Historical, Romance, Supernatural, Political",8.4
684,Queen: Love and War,"Historical, Romance, Fantasy, Political",7.9
776,Grand Prince,"Historical, Romance, Drama, Political",7.8
80,Arthdal Chronicles Part 2: The Sky Turning Ins...,"Historical, Romance, Fantasy, Political",8.7
72,Arthdal Chronicles Part 3: The Prelude to All ...,"Historical, Romance, Fantasy, Political",8.7
847,Moon in the Day,"Thriller, Historical, Romance, Fantasy",7.7
156,Perfect Marriage Revenge,"Business, Romance, Drama, Fantasy",8.5
411,Bloody Heart,"Historical, Drama, Melodrama, Political",8.2
848,The King in Love,"Historical, Romance, Drama, Political",7.7
600,Captivating the King,"Historical, Romance, Drama, Melodrama",8.0


In [18]:
data[data['title'].str.contains('Scarlet')]

,id,title,native-title,genre,director,format,type,country,release_date,duration,...,synopsis,duration (min),release_year,rate_num,score,user_count,rating_norm,actor,combined_emb,combined_tf
57,15999,Moon Lovers: Scarlet Heart Ryeo,달의 연인 - 보보경심 려,"Historical, Romance, Fantasy, Melodrama",Kim Kyu Tae,Standard Series,Drama,South Korea,"Aug 29, 2016",60 min.,...,"When a total eclipse of the sun takes place, G...",60,2016,15,8.7,92862.0,0.87,"Lee Joon Gi, IU, Kang Ha Neul, Hong Jong Hyun,...","Moon Lovers: Scarlet Heart Ryeo Historical, Ro...",moon lovers scarlet heart ryeo historical roma...
2061,790396,tvN O'PENing: Hwa Ja’s Scarlet,화자의 스칼렛,Drama,NaN,Drama Special,Drama,South Korea,"Oct 3, 2025",NaN,...,"Tells the story of a daughter, who was given u...",0,2025,0,0.0,0.0,0.00,"Oh Na Ra, Seo Young Hee, Kim Si Eun, Hwang Seo...",tvN O'PENing: Hwa Ja’s Scarlet Drama Drama Fam...,tvn o pening hwa ja s scarlet drama drama fami...
3264,5893,The Scarlet Letter,주홍글씨,"Romance, Drama, Melodrama",Lee Min Soo,Standard Series,Drama,South Korea,"Aug 9, 2010",40 min.,...,Four people paying the consequences for a love...,40,2010,0,7.2,24.0,0.72,"Lee Seung Yeon, Kim Yun Joo, Kim Young Ho, Jo ...","The Scarlet Letter Romance, Drama, Melodrama R...",the scarlet letter romance drama melodrama rom...


# Build Backend

In [11]:
from fastapi import FastAPI, Query
# from annoy import AnnoyIndex
# import pandas as pd
# import numpy as np
from typing import Optional
import pickle

In [12]:
app = FastAPI()

data = pd.read_csv('data/clean/movies_drama_v2.csv')
tfidf_sim = np.load('artifact/tfidf_sim_v2.npy')
annoy_index = AnnoyIndex(384, 'angular')
annoy_index.load('artifact/index_movdrama.ann')

with open('artifact/vectorizer.pkl', "rb") as f:
    vectorizer = pickle.load(f)

MIN_RATING = 5.5

In [22]:
# @app.get("recommend/{drama_idx}")
def recommend(
    drama_idx: int,
    top_n: int              = 10,
    w_tfidf: float          = 0.3,
    w_minilm: float         = 0.5,
    w_rating: float         = 0.2,
    genre: Optional[str]    = None,
    year_min: Optional[int] = None,
    candidate_pool: int     =200
):
    #scoring
    ### TF-IDF
    tfidf_scores = tfidf_sim[drama_idx]

    ### Annoy Index
    neighbor_ids, distances = annoy_index.get_nns_by_item(drama_idx, candidate_pool, include_distances=True)
    minilm_scores = np.zeros(len(data))
    for idx, distance in zip(neighbor_ids, distances):
        minilm_scores[idx] = 1 - (distance**2)/2

    #Final score for recommendation
    final_scores = (
        w_tfidf * tfidf_scores +
        w_minilm * minilm_scores +
        w_rating * data['rating_norm'].values
    )

    ## Exclude query drama
    final_scores[drama_idx] = -1

    ## Get Top-N
    top_idx = final_scores.argsort()[::-1][:top_n]
    results = data.iloc[top_idx][[
        "title", "genre", "score", "release_year", "synopsis"
    ]].to_dict(orient= "records")

    return {
        "query": data.iloc[drama_idx]["title"],
        "recommendations": results
    }

In [23]:
recommend(1)

{'query': 'Twinkling Watermelon',
 'recommendations': [{'title': 'Begins Youth',
   'genre': 'Life, Youth, Drama, Fantasy',
   'score': 8.6,
   'release_year': 2024,
   'synopsis': 'The drama follows seven boys navigating school and personal growth, each facing challenges like family struggles, loss, poverty, and rejection. After returning to Songju-si, Kim Hwan meets six boys who initially view him with suspicion, but over time, they become close friends, bonding over shared hardships. As they support each other, Hwan grows fond of Songju-si and wishes to stay. However, his father, Kim Chang Jun, disapproves of the friendship and demands Hwan distance himself, leading Hwan to confront his emotions and stand up for himself.'},
  {'title': '18 Again',
   'genre': 'Romance, Life, Drama, Fantasy',
   'score': 8.6,
   'release_year': 2020,
   'synopsis': "Jung Da Jeong and Hong Dae Yeong have been married for almost two decades, raising their twins together. Despite appearances, their life